<div dir="rtl" align="right">
<h1>فاز دوم پروژه آمار: احتمال و توزیع‌های احتمالی 
<hr>
</div>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

# ۱. بارگذاری و ادغام ۸ فایل موضوعی   
data_dir = Path("../data")

lux_df = pd.read_sas(data_dir / "LUX_L.xpt", format="xport")
lux_cols = [c for c in ["SEQN", "LUAXSTAT", "LUARXNC", "LUANMVGP", "LUXSMED", "LUXSIQR", "LUXSIQRM", "LUXCAPM"] if c in lux_df.columns]
df_merged = lux_df[lux_cols].copy()

other_files_vars = {
    "DEMO_L.xpt": ["SEQN", "RIDAGEYR", "RIAGENDR", "WTMEC2YR"],
    "CBC_L.xpt": ["SEQN", "LBDNENO", "LBDLYMNO"],
    "BMX_L.xpt": ["SEQN", "BMXBMI"],
    "BPXO_L.xpt": ["SEQN", "BPXOSY1"],
    "GHB_L.xpt": ["SEQN", "LBXGH"],
    "TCHOL_L.xpt": ["SEQN", "LBXTC"],
    "BIOPRO_L.xpt": ["SEQN", "LBXSATSI", "LBXSASSI"]
}

for fname, cols in other_files_vars.items():
    fpath = data_dir / fname
    temp_df = pd.read_sas(fpath, format="xport")[cols]
    df_merged = pd.merge(df_merged, temp_df, on="SEQN", how="right" if fname == "DEMO_L.xpt" else "left")

# ۲. محاسبه شاخص التهاب سیستمیک و اعمال فیلترهای پایه فاز اول
df_merged["NLR"] = np.where(
    df_merged["LBDLYMNO"] > 0,
    df_merged["LBDNENO"] / df_merged["LBDLYMNO"],
    np.nan
)

n_total = len(df_merged)
df_adults = df_merged[df_merged["RIDAGEYR"] >= 20].copy()
n_under20 = n_total - len(df_adults)

df_mec = df_adults[df_adults["WTMEC2YR"] > 0].copy()
n_no_mec = len(df_adults) - len(df_mec)

df_phase1 = df_mec.dropna(subset=["LUXSMED", "NLR"]).copy()
n_missing_main = len(df_mec) - len(df_phase1)

# ۳. اعمال ۳ پارامتر کنترل کیفیت الاستوگرافی کبد (ناشتایی، تعداد دفعات تست، واریانس/تغییرپذیری)
n_fasting_fail = int((df_phase1["LUARXNC"] == 1).sum())
n_measures_fail = int(df_phase1["LUANMVGP"].isin([1, 2]).sum())
n_var_fail = int((df_phase1["LUXSIQRM"] >= 30).sum())

cond_quality = (
    (df_phase1["LUAXSTAT"] == 1) &
    (~df_phase1["LUANMVGP"].isin([1, 2])) &
    (df_phase1["LUXSIQRM"] < 30)
)

df_qc = df_phase1[cond_quality].copy()
n_qc_dropped = len(df_phase1) - len(df_qc)

# ۴. حذف ردیف‌های دارای داده گمشده در متغیرهای فرعی و کنترلی
secondary_cols = ["LUXCAPM", "BMXBMI", "BPXOSY1", "LBXGH", "LBXTC", "LBXSATSI", "LBXSASSI"]
all_analysis_cols = ["RIDAGEYR", "RIAGENDR", "WTMEC2YR", "LUXSMED", "NLR"] + secondary_cols

df_clean = df_qc.dropna(subset=all_analysis_cols).copy().reset_index(drop=True)
n_sec_dropped = len(df_qc) - len(df_clean)

print("\n=== جریان گام‌به‌گام تعیین حجم نمونه تحلیلی نهایی (فاز دوم - موارد کاملاً بدون داده گمشده) ===")
print(f"\u202b۱. تعداد کل شرکت‌کنندگان پس از ادغام ۸ فایل: {n_total} نفر\u202c")
print(f"\u202b۲. حذف افراد زیر ۲۰ سال: {n_under20} نفر (باقی‌مانده بزرگسالان: {len(df_adults)} نفر)\u202c")
print(f"\u202b۳. حذف بزرگسالان فاقد وزن معاینه (WTMEC2YR = 0): {n_no_mec} نفر (باقی‌مانده: {len(df_mec)} نفر)\u202c")
print(f"\u202b۴. حذف به دلیل داده گمشده در متغیرهای اصلی (LUXSMED یا NLR): {n_missing_main} نفر (نمونه فاز اول: {len(df_phase1)} نفر)\u202c")
print(f"\u202b۵. حذف به دلیل عدم احراز ۳ پارامتر کیفیت الاستوگرافی کبد: {n_qc_dropped} نفر (باقی‌مانده باکیفیت: {len(df_qc)} نفر)\u202c")
print(f"\u202b   - پارامتر ۱ (زمان ناشتایی کمتر از ۳ ساعت | LUARXNC == 1): {n_fasting_fail} نفر\u202c")
print(f"\u202b   - پارامتر ۲ (تعداد اندازه‌گیری معتبر کمتر از ۱۰ بار): {n_measures_fail} نفر\u202c")
print(f"\u202b   - پارامتر ۳ (تغییرپذیری بالای اندازه‌گیری‌ها | IQR/Median >= 30%): {n_var_fail} نفر\u202c")
print(f"\u202b۶. حذف به دلیل داده گمشده در متغیرهای فرعی و کنترلی: {n_sec_dropped} نفر\u202c")
print(f"\u202b==> ۷. حجم نمونه تحلیلی نهایی و کاملاً بدون داده گمشده (df_clean): {len(df_clean)} نفر\u202c\n")


=== جریان گام‌به‌گام تعیین حجم نمونه تحلیلی نهایی (فاز دوم - موارد کاملاً بدون داده گمشده) ===
‫۱. تعداد کل شرکت‌کنندگان پس از ادغام ۸ فایل: 11933 نفر‬
‫۲. حذف افراد زیر ۲۰ سال: 4124 نفر (باقی‌مانده بزرگسالان: 7809 نفر)‬
‫۳. حذف بزرگسالان فاقد وزن معاینه (WTMEC2YR = 0): 0 نفر (باقی‌مانده: 7809 نفر)‬
‫۴. حذف به دلیل داده گمشده در متغیرهای اصلی (LUXSMED یا NLR): 2449 نفر (نمونه فاز اول: 5360 نفر)‬
‫۵. حذف به دلیل عدم احراز ۳ پارامتر کیفیت الاستوگرافی کبد: 319 نفر (باقی‌مانده باکیفیت: 5041 نفر)‬
‫   - پارامتر ۱ (زمان ناشتایی کمتر از ۳ ساعت | LUARXNC == 1): 100 نفر‬
‫   - پارامتر ۲ (تعداد اندازه‌گیری معتبر کمتر از ۱۰ بار): 17 نفر‬
‫   - پارامتر ۳ (تغییرپذیری بالای اندازه‌گیری‌ها | IQR/Median >= 30%): 179 نفر‬
‫۶. حذف به دلیل داده گمشده در متغیرهای فرعی و کنترلی: 427 نفر‬
‫==> ۷. حجم نمونه تحلیلی نهایی و کاملاً بدون داده گمشده (df_clean): 4614 نفر‬



<div dir="rtl" align="right">
<h3>جمع‌بندی آماده‌سازی داده، تعیین حجم نمونه نهایی و علت حذف داده‌های گمشده</h3>
در فاز اول پروژه، با اعمال شرط سنی (بزرگسالان ۲۰ سال و بالاتر)، داشتن وزن معاینه معتبر (<code>WTMEC2YR > 0</code>) و حذف مقادیر گمشده در دو متغیر اصلی (<code>LUXSMED</code> و <code>NLR</code>)، نمونه اولیه شامل ۵,۳۶۰ شرکت‌کننده به دست آمد.<br><br>
به منظور افزایش دقت اندازه‌گیری و یکدست‌سازی کامل فضای نمونه در تمامی تحلیل‌ها، دو مرحله پالایش تکمیلی اعمال گردید:<br>
<b>۱. اعمال معیارهای کنترل کیفیت الاستوگرافی کبد:</b> ۳۱۹ مشاهده که فاقد معیارهای کیفیت معاینه کامل (<code>LUAXSTAT == 1</code>) بودند — به دلیل ناشتایی کمتر از ۳ ساعت (<code>LUARXNC == 1</code> شامل ۱۰۰ نفر)، تعداد دفعات اندازه‌گیری معتبر کمتر از ۱۰ بار (<code>LUANMVGP</code> با کدهای ۱ و ۲ شامل ۱۷ نفر) یا تغییرپذیری و واریانس بالای اندازه‌گیری‌ها (<code>LUXSIQRM >= 30%</code> شامل ۱۷۹ نفر) — کنار گذاشته شدند تا ۵,۰۴۱ مشاهده باکیفیت باقی بمانند.<br>
<b>۲. حذف کامل مقادیر گمشده در متغیرهای فرعی و کنترلی:</b> به منظور انجام تحلیل موارد کامل (Complete-Case Analysis) روی یک فضای نمونه واحد، ۴۲۷ مشاهده که در هر یک از متغیرهای فرعی (<code>LUXCAPM</code>، <code>BMXBMI</code>، <code>BPXOSY1</code>، <code>LBXGH</code>، <code>LBXTC</code>، <code>LBXSATSI</code> و <code>LBXSASSI</code>) دارای مقدار گمشده بودند نیز حذف گردیدند تا در مجموعه داده تحلیلی نهایی (<code>df_clean</code>) هیچ مقدار گمشده‌ای در هیچ متغیری وجود نداشته باشد.<br><br>
<b>دلایل آماری و علمی حذف داده‌های گمشده (روش تحلیل موارد کامل):</b><br>
• <b>یکپارچگی فضای نمونه (Ω) در نظریه احتمال:</b> در محاسبات فاز دوم (شامل احتمال‌های حاشیه‌ای، مشترک، شرطی، قضیه بیز و کوواریانس)، تمامی رویدادها و متغیرهای تصادفی باید بر روی یک فضای نمونه واحد و ثابت تعریف شوند. وجود داده‌های گمشده باعث تغییر مخرج کسر (N) از یک متغیر به متغیر دیگر شده و سازگاری ریاضی روابط احتمالی (مانند <code>P(A ∩ B)</code> و قانون احتمال کل) را مختل می‌کند.<br>
• <b>جلوگیری از اریبی و تحریف توزیع‌های تجربی:</b> در متغیرهای زیستی و آزمایشگاهی (مانند سفتی کبد، شاخص التهاب و آنزیم‌های کبدی)، جایگذاری مقادیر گمشده با مقادیر مصنوعی (نظیر میانگین یا میانه) باعث کاهش کاذب واریانس، تغییر شکل واقعی چولگی توزیع‌ها و ایجاد خطا در برآورد احتمال رویدادهای حدی و رابطه واقعی بین متغیرها می‌شود.<br>
• <b>کفایت بالای حجم نمونه باقی‌مانده:</b> پس از حذف موارد ناقص، حجم نمونه ۴,۶۱۴ نفری همچنان بسیار بزرگ بوده و از توان و دقت آماری کاملاً کافی برای برآورد توزیع‌های تجربی و پارامترهای احتمالی برخوردار است.
<hr>
</div>

<div dir="rtl" align="right">
<h2>تسک مشترک ۱: تعریف رویدادها و متغیرهای تصادفی</h2>
<b>تعریف واحد مشاهده و فضای نمونه (Ω):</b><br> پس از اعمال فیلترهای کنترل کیفیت الاستوگرافی کبد و حذف کامل تمامی مقادیر گمشده ، فضای نمونه شامل <b>۴,۶۱۴ ردیف</b> موجود در مجموعه داده تحلیلی نهایی (<code>df_clean</code>) است. هر <b>مشاهده</b> در این فضای نمونه، دقیقاً متناظر با <b>یک شرکت‌کننده بزرگسال مستقل (۲۰ سال و بالاتر) دارای معاینه معتبر در پیمایش NHANES (۲۰۲۱–۲۰۲۳)</b> است که در مجموعه داده با شناسه یکتای <code>SEQN</code> در یک سطر مجزا مشخص می‌شود.<br><br>
<b>الف) تعریف رویدادهای تصادفی :</b><br>
• <b>رویداد A (ابتلا به فیبروز پیشرفته کبدی - Advanced Fibrosis):</b> شامل تمام مشاهداتی (ردیف‌هایی با شناسه <code>SEQN</code>) است که در ستون سفتی کبد (<code>LUXSMED</code> از فایل <code>LUX_L</code>) دارای مقدار برابر یا بیشتر از آستانه بالینی ۸.۲ کیلوپاسکال باشند (<code>LUXSMED >= 8.2</code>). وقوع این رویداد در دیتافریم با ستون دودویی <code>Event_A_Fibrosis</code> (مقدار <code>1</code> برای وقوع و <code>0</code> برای عدم وقوع در آن مشاهده) ثبت شده است.<br>
• <b>رویداد B (التهاب سیستمیک بالا - High NLR):</b> شامل تمام مشاهداتی (ردیف‌هایی با شناسه <code>SEQN</code>) است که در متغیر نسبت نوتروفیل به لنفوسیت (<code>NLR</code> حاصل از فایل <code>CBC_L</code>) دارای مقدار برابر یا بیشتر از آستانه بالینی ۳.۰ باشند (<code>NLR >= 3.0</code>). این رویداد با ستون دودویی <code>Event_B_HighNLR</code> (مقدار <code>1</code> برای وقوع و <code>0</code> برای عدم وقوع) مشخص می‌شود.<br>
• <b>رویدادهای زیرگروهی G (جنسیت مرد) و S (سالمندی ۶۰ سال و بالاتر):</b> برای تحلیل احتمال‌های شرطی در زیرگروه‌ها، رویداد <code>G</code> شامل مشاهداتی است که در ستون <code>RIAGENDR</code> دارای کد <code>1</code> (مرد) هستند (<code>Event_G_Male</code>) و رویداد <code>S</code> شامل مشاهداتی است که در ستون سن (<code>RIDAGEYR</code>) دارای مقدار ۶۰ سال و بالاتر هستند (<code>Event_S_Senior</code>).<br><br>
<b>ب) تعریف متغیرهای تصادفی :</b><br>
• <b>متغیر تصادفی پیوسته Y (سفتی کبد):</b> تابعی است که به هر مشاهده (شرکت‌کننده با شناسه <code>SEQN</code>)، مقدار عددی پیوسته ثبت‌شده در ستون <code>LUXSMED</code> همان ردیف را بر حسب کیلوپاسکال (<code>kPa</code>) نسبت می‌دهد.<br>
• <b>متغیر تصادفی پیوسته X (شاخص التهاب سیستمیک):</b> تابعی است که به هر مشاهده (شرکت‌کننده با شناسه <code>SEQN</code>)، مقدار عددی پیوسته ثبت‌شده در ستون <code>NLR</code> همان ردیف را نسبت می‌دهد.<br>
• <b>متغیر تصادفی گسسته برنولی I_A (نشانگر فیبروز پیشرفته):</b> تابعی گسسته با مقادیر <code>{0, 1}</code> است که به هر مشاهده (ردیف <code>SEQN</code>) در صورت وقوع رویداد فیبروز پیشرفته (<code>LUXSMED >= 8.2</code>) عدد <code>1</code> و در غیر این صورت عدد <code>0</code> را نسبت می‌دهد.
</div>